In [3]:
# Standard libraries
import os

# Matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# OpenCV
import cv2

## Constants

In [4]:
SOURCE_IMAGE_DIR = ".data/eval/"
GRAYSCALE_IMAGE_DIR = ".data/eval_grayscale"

CATEGORY_LABELS = [
    "not_food",
    "italian_food",
    "japanese_food",
    "meat",
    "seafood",
    "soup",
    "salad",
    "dessert"
]

## Generate Grayscale Images

In [6]:
CATEGORY_LABELS = ['.']

for img_dir in CATEGORY_LABELS:
    sample_count = len(os.listdir(os.path.join(SOURCE_IMAGE_DIR, img_dir)))
    print(f"Processing category: {img_dir} ({sample_count} samples)")
    for filename in os.listdir(os.path.join(SOURCE_IMAGE_DIR, img_dir)):
        if not (filename.endswith(".jpg") or filename.endswith(".png")): continue
        
        # Processing
        img = cv2.imread(os.path.join(SOURCE_IMAGE_DIR, img_dir, filename), cv2.IMREAD_GRAYSCALE)
        img_resized = cv2.resize(img, (300, 300), cv2.INTER_AREA)

        grayscale_dir = os.path.join(GRAYSCALE_IMAGE_DIR, f"{img_dir}")
        os.makedirs(grayscale_dir, exist_ok=True)
        cv2.imwrite(os.path.join(grayscale_dir, filename), img_resized)

Processing category: . (5 samples)


In [ ]:
X, y = [], []

for img_dir in CATEGORY_LABELS:
    sample_count = len(os.listdir(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir)))
    print(f"Importing category: {img_dir} ({sample_count} samples)")

    for filename in os.listdir(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir)):
        if not (filename.endswith(".jpg") or filename.endswith(".png")): continue

        img = cv2.imread(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir, filename), cv2.IMREAD_GRAYSCALE)
        X.append(img)
        y.append(CATEGORY_LABELS.index(img_dir))

print("------------------------------")
print(f"Data imported. Total samples: {len(X)}")

Importing category: not_food (4319 samples)
Importing category: italian_food (2000 samples)
Importing category: japanese_food (3000 samples)
Importing category: meat (6000 samples)
Importing category: seafood (2000 samples)
Importing category: soup (998 samples)
Importing category: salad (1000 samples)
Importing category: dessert (5000 samples)
------------------------------
Data imported. Total samples: 24317


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from cnn import ResNet, Bottleneck

net = ResNet(Bottleneck, [2, 2, 2, 2], num_classes=len(CATEGORY_LABELS))
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

X_tensor = torch.tensor(X, dtype=torch.float32)  # Add channel dimension
y_tensor = torch.tensor(y, dtype=torch.int16)
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [10]:
for epoch in range(5):
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:
            print(f"[{epoch + 1}, {i + 1}] loss: {running_loss / 200:.3f}")
            running_loss = 0.0

print("Finished Training") 

RuntimeError: Given groups=1, weight of size [64, 3, 7, 7], expected input[32, 1, 300, 300] to have 3 channels, but got 1 channels instead

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier
import time

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

confusion_matrices = []
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    start = time.time()
    X_train, X_test = [X[i] for i in train_idx], [X[i] for i in test_idx]
    y_train, y_test = [y[i] for i in train_idx], [y[i] for i in test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    confusion_matrices.append(confusion_matrix(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred, average='macro'))
    print(classification_report(y_test, y_pred))
    print(f"Finished fold: {fold + 1} ({time.time() - start:.2f} seconds)")


print("Average F1 Score:", sum(f1_scores) / len(f1_scores))

ValueError: Found array with dim 3, while dim <= 2 is required by RandomForestClassifier.